# E3 -- extended Muller-Brown (10D): run notebook

**This notebook runs and saves. It does not typeset figures.**

Every official metric is computed here, at run time, and written into each run's `metrics_timeseries.csv` and `cost_timeseries.csv`. The companion notebook `E3_muller_brown_plot.ipynb` reads those numbers and never recomputes them.

**Run All executes the single default full configuration.** There is exactly one configuration for this experiment, `configs/experiments/E3.yaml` -- there is no smoke, dev, reduced, or production profile to choose between. Lowering the particle count for local debugging is an explicit temporary edit, never a second committed profile.

Each variant is saved the moment it finishes, into its own atomically renamed run directory, so a variant that fails leaves the earlier ones untouched.

In [ ]:
import sys

sys.path.insert(0, "..")  # importable when launched from notebooks/

from src.pipeline import load_experiment, run_variants_and_save

## Target, reference, and cost calibration

The reference is built **once** and reused by every method. It does not depend on any method parameter, so it is **never rebuilt per method, per hyperparameter value, or per canonical/tamed variant**; a cached reference on disk is loaded instead of being recomputed.

The force-equivalent-evaluation (FEE) calibration is measured once per device in the same way, and every run in this experiment is costed against that one calibration. The device is resolved automatically -- no device index is pinned in this notebook.

In [ ]:
experiment = load_experiment("E3", device="auto")

reference = experiment.ensure_reference()
fee = experiment.ensure_fee_calibration()

described = reference.describe()
print(f"reference: kind={described.get('kind', described.get('method'))}  hash={experiment.reference_hash}")
print(f"FEE:       unit={fee.cost_unit}  hash={fee.hash}")

## ULA

Every taming-capable method runs **both** a canonical and a tamed variant. `run_variants_and_save` expands each entry of `variants` into those two runs by itself, so a notebook never passes `tame`. The two variants are **calibrated separately** -- each one gets its own step size from its own `dt` refinement -- and each is saved as its own run directory.

In [ ]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="ULA",
    # `run_variants_and_save` expands each entry below into a
    # canonical and a tamed run, so `tame` is never passed here.
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

## MALA

MALA supports taming, so it also runs both variants. Tamed MALA implements the actual tamed proposal density in the Metropolis-Hastings ratio; it is a genuine second sampler, not a relabelled copy of the canonical run.

In [ ]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="MALA",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

## FLA

The three stability indices are this experiment's default grid in `configs/registry.yaml`. All three run from this one cell and save as separate variants, and each of them is expanded into a canonical and a tamed run.

In [ ]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="FLA",
    variants=[
        {"alpha": 1.6}, {"alpha": 1.7}, {"alpha": 1.8},
    ],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

## ULD

ULD is the method; BAOAB is the integrator it is discretised with. Runs, manifests, and legends say ULD.

In [ ]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="ULD",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

## PT

Parallel tempering. The replica ladder is tuned by the calibration step that `run_variants_and_save` invokes, not here, and the tuned ladder is written into the run's `calibration.json`.

In [ ]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="PT",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

## Raw-CP

The same compound-Poisson jump process with the Levy score correction switched off. It does not preserve the target, so it is the control arm that isolates what the score correction buys, not a competitive baseline.

In [ ]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="Raw-CP",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

## LSC-CP

Compound-Poisson jumps with the full deterministic-quadrature Levy score correction.

In [ ]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="LSC-CP",
    variants=[{}],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

## LSC-CP-RA

`A` is the **iid Monte Carlo bank size of one estimator family**, LSC-CP-RA. `A = 1, 4, 8` are variants of that single family, not three separate methods, and **all of them run from this one cell** and save as separate variants.

The bank holds `A` displacements drawn iid from the full normalised jump law `rho = nu / lambda`, and **the same bank drives both the score and the compound-Poisson increment**.

In [ ]:
RUN_PRODUCTION = True
RUN_STATIONARITY = False

run_variants_and_save(
    experiment=experiment,
    method="LSC-CP-RA",
    variants=[
        {"A": 1}, {"A": 4}, {"A": 8},
    ],
    run_production=RUN_PRODUCTION,
    run_stationarity=RUN_STATIONARITY,
)

## Rebuild the catalog

`catalog.csv` is a **derived index** over the run manifests. It is never written by a worker mid-run, so concurrent runs never contend for it, and it can be rebuilt at any time by rescanning the manifests -- a lost or stale catalog costs nothing. Only runs that verify (manifest present, `COMPLETE` present, hashes matching) are admitted.

In [ ]:
from src.catalog import write_catalog

report = write_catalog(experiment.paths.experiment_dir)
print(f"catalog rebuilt: {report['n_runs']} runs, {report['n_rejected']} rejected")